# B5: PEFT LoRA / QLoRA Fine-Tuning — Qwen2.5-1.5B-Instruct (PRDECT-ID)
**Penanggung Jawab:** Fai (Reasoning & Adaptation Lead)  
**Tujuan:** Melatih adapter LoRA/QLoRA (`r=16, lora_alpha=32`) pada base generator yang sama (`Qwen/Qwen2.5-1.5B-Instruct`) menggunakan `data/processed/train.csv` (bebas kebocoran dev/test) untuk Konfigurasi **B5** dan **Ablasi A4 (In-Domain vs OOD Category Drift)**.

In [ ]:
!pip install -q transformers peft trl bitsandbytes datasets accelerate scikit-learn pandas

In [ ]:
import json
import torch
import pandas as pd
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
OUTPUT_DIR = "models/qwen25_15b_prdect_lora"
SEED = 42

train_df = pd.read_csv("data/processed/train.csv")
print(f"Loaded {len(train_df)} training reviews from train.csv")

In [ ]:
def format_instruction(row):
    text = str(row['Customer Review']).strip()
    prompt = (
        "Analisis sentimen (Positive/Negative/ABSTAIN) dan emosi (Happy/Love/Sadness/Fear/Anger/ABSTAIN) "
        f"ulasan produk berikut:\nUlasan: \"{text}\"\nOutput JSON:"
    )
    target = json.dumps({
        "sentiment": str(row['Sentiment']),
        "emotion": str(row['Emotion']),
        "guideline_version": "v1.0",
        "abstain": False,
        "abstain_reason": None
    }, ensure_ascii=False)
    return {"text": f"<|im_start|>user\n{prompt}<|im_end|>\n<|im_start|>assistant\n{target}<|im_end|>"}

train_ds = Dataset.from_list([format_instruction(r) for _, r in train_df.iterrows()])
print(train_ds[0]['text'])

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    num_train_epochs=2,
    logging_steps=25,
    save_strategy="epoch",
    fp16=True,
    seed=SEED,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_ds,
    dataset_text_field="text",
    max_seq_length=512,
    tokenizer=tokenizer,
    args=training_args,
)

trainer.train()
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"LoRA adapter saved to {OUTPUT_DIR}")